# Customer Churn Prediction with 5 ML Models



## 1. Imports


In [16]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


## 2. Load dataset




In [17]:
DATA_PATH = Path("archive/telco.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


Shape: (7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


## 3. Cleaning and feature selection


- **Target leakage:** `Customer Status`, `Churn Score`, `Churn Category`, `Churn Reason`, `Satisfaction Score`

`Offer` and `Internet Type` nulls are filled with `None` (service not applicable).


In [18]:
leakage_or_id_cols = [
    "Customer ID",
    "Customer Status",
    "Churn Score",
    "Churn Category",
    "Churn Reason",
   
]

df = df.drop(columns=[c for c in leakage_or_id_cols if c in df.columns])

for col in ["Offer", "Internet Type"]:
    if col in df.columns:
        df[col] = df[col].fillna("None")

print("Missing values after cleaning:", int(df.isnull().sum().sum()))
print("Columns:", df.columns.tolist())


Missing values after cleaning: 0
Columns: ['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Country', 'State', 'City', 'Zip Code', 'Latitude', 'Longitude', 'Population', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'Churn Label', 'CLTV']


In [19]:
y = df["Churn Label"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn Label"])

categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of input features:", X.shape[1])
print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)
print("Churn rate:", round(y.mean(), 4))



Number of input features: 44
Categorical: ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Country', 'State', 'City', 'Quarter', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']
Numerical: ['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'CLTV']
Churn rate: 0.2654


## 4. Preprocessing and train/test split


In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

# Dense encoding for GaussianNB (does not accept sparse matrices)
gnb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


Train size: 5634 | Test size: 1409


## 5. Helper: evaluate a model


In [21]:
def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "AUC": roc_auc_score(y_te, y_prob),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1": f1_score(y_te, y_pred),
        "MCC": matthews_corrcoef(y_te, y_pred),
    }

    print(f"\n========== {name} ==========\n")
    for k in ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]:
        print(f"{k:10s}: {metrics[k]:.4f}")

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_te, y_pred))
    print("\nClassification Report\n")
    print(classification_report(y_te, y_pred))
    return metrics, y_pred


## 6. Logistic Regression


In [22]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
lr_model.fit(X_train, y_train)
lr_metrics, _ = evaluate_model("Logistic Regression", lr_model, X_test, y_test)



========== Logistic Regression ==========

Accuracy  : 0.9617
AUC       : 0.9918
Precision : 0.9545
Recall    : 0.8984
F1        : 0.9256
MCC       : 0.9006

Confusion Matrix

[[1019   16]
 [  38  336]]

Classification Report

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1035
           1       0.95      0.90      0.93       374

    accuracy                           0.96      1409
   macro avg       0.96      0.94      0.95      1409
weighted avg       0.96      0.96      0.96      1409



## 7. Decision Tree


In [23]:
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
    )),
])
dt_model.fit(X_train, y_train)
dt_metrics, _ = evaluate_model("Decision Tree", dt_model, X_test, y_test)



========== Decision Tree ==========

Accuracy  : 0.9517
AUC       : 0.9858
Precision : 0.9554
Recall    : 0.8583
F1        : 0.9042
MCC       : 0.8743

Confusion Matrix

[[1020   15]
 [  53  321]]

Classification Report

              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1035
           1       0.96      0.86      0.90       374

    accuracy                           0.95      1409
   macro avg       0.95      0.92      0.94      1409
weighted avg       0.95      0.95      0.95      1409



## 8. K-Nearest Neighbors


In [24]:
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=11)),
])
knn_model.fit(X_train, y_train)
knn_metrics, _ = evaluate_model("KNN", knn_model, X_test, y_test)



========== KNN ==========

Accuracy  : 0.9219
AUC       : 0.9630
Precision : 0.8952
Recall    : 0.7995
F1        : 0.8446
MCC       : 0.7950

Confusion Matrix

[[1000   35]
 [  75  299]]

Classification Report

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1035
           1       0.90      0.80      0.84       374

    accuracy                           0.92      1409
   macro avg       0.91      0.88      0.90      1409
weighted avg       0.92      0.92      0.92      1409



## 9. Gaussian Naive Bayes


In [25]:
gnb_model = Pipeline([
    ("preprocessor", gnb_preprocessor),
    ("classifier", GaussianNB()),
])
gnb_model.fit(X_train, y_train)
gnb_metrics, _ = evaluate_model("Gaussian Naive Bayes", gnb_model, X_test, y_test)



========== Gaussian Naive Bayes ==========

Accuracy  : 0.4301
AUC       : 0.5199
Precision : 0.2768
Recall    : 0.7112
F1        : 0.3985
MCC       : 0.0377

Confusion Matrix

[[340 695]
 [108 266]]

Classification Report

              precision    recall  f1-score   support

           0       0.76      0.33      0.46      1035
           1       0.28      0.71      0.40       374

    accuracy                           0.43      1409
   macro avg       0.52      0.52      0.43      1409
weighted avg       0.63      0.43      0.44      1409



## 10. Random Forest (Ensemble)


In [26]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])
rf_model.fit(X_train, y_train)
rf_metrics, _ = evaluate_model("Random Forest", rf_model, X_test, y_test)



========== Random Forest ==========

Accuracy  : 0.8836
AUC       : 0.9540
Precision : 0.9688
Recall    : 0.5802
F1        : 0.7258
MCC       : 0.6925

Confusion Matrix

[[1028    7]
 [ 157  217]]

Classification Report

              precision    recall  f1-score   support

           0       0.87      0.99      0.93      1035
           1       0.97      0.58      0.73       374

    accuracy                           0.88      1409
   macro avg       0.92      0.79      0.83      1409
weighted avg       0.89      0.88      0.87      1409



## 11. Model comparison table


In [27]:
results_df = pd.DataFrame([
    lr_metrics,
    dt_metrics,
    knn_metrics,
    gnb_metrics,
    rf_metrics,
])

display_cols = ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison = results_df[display_cols].copy()
for col in display_cols[1:]:
    comparison[col] = comparison[col].round(4)

print("\n===== Comparison Table =====\n")
print(comparison.to_string(index=False))

best_row = results_df.loc[results_df["F1"].idxmax()]
print(
    f"\nOverall winner (by F1): {best_row['Model']} "
    f"(F1={best_row['F1']:.4f}, AUC={best_row['AUC']:.4f}, MCC={best_row['MCC']:.4f})"
)



===== Comparison Table =====

               Model  Accuracy    AUC  Precision  Recall     F1    MCC
 Logistic Regression    0.9617 0.9918     0.9545  0.8984 0.9256 0.9006
       Decision Tree    0.9517 0.9858     0.9554  0.8583 0.9042 0.8743
                 KNN    0.9219 0.9630     0.8952  0.7995 0.8446 0.7950
Gaussian Naive Bayes    0.4301 0.5199     0.2768  0.7112 0.3985 0.0377
       Random Forest    0.8836 0.9540     0.9688  0.5802 0.7258 0.6925

Overall winner (by F1): Logistic Regression (F1=0.9256, AUC=0.9918, MCC=0.9006)


### Observations

| ML Model | Observation |
|---|---|
| Logistic Regression | Best overall on this hold-out set (highest F1, AUC, and MCC). Strong calibrated linear baseline after scaling + one-hot encoding. |
| Decision Tree | Interpretable; constrained depth reduces overfit, but slightly behind LR/RF on F1 and AUC. |
| KNN | Competitive after StandardScaler; still sensitive to class imbalance and feature space size. |
| Gaussian Naive Bayes | Highest Recall but lowest Precision — flags many churners, with more false alarms. Independence assumption limits F1/MCC. |
| Random Forest | Tied-best Accuracy and strong Precision/AUC; slightly behind LR on F1/Recall for the churn class. |
| **Overall Winner** | **Logistic Regression** (best F1 / AUC / MCC on the test set). |



## 12. Save models and test data (for Streamlit)


In [28]:
os.makedirs("model", exist_ok=True)

joblib.dump(lr_model, "model/logistic_regression.joblib")
joblib.dump(dt_model, "model/decision_tree.joblib")
joblib.dump(knn_model, "model/knn.joblib")
joblib.dump(gnb_model, "model/naive_bayes.joblib")
joblib.dump(rf_model, "model/random_forest.joblib")

# Test CSV includes features + true label for Streamlit evaluation
test_export = X_test.copy()
test_export["Churn Label"] = y_test.map({1: "Yes", 0: "No"}).values
test_export.to_csv("test_data.csv", index=False)

comparison.to_csv("model/metrics_comparison.csv", index=False)

print("Saved models to model/")
print("Saved test_data.csv with", len(test_export), "rows")
print("Saved model/metrics_comparison.csv")


Saved models to model/
Saved test_data.csv with 1409 rows
Saved model/metrics_comparison.csv
